## 2 Global Navigation

Authored by: Jiri Blaha

### [2A] Module Overview

The global navigation module transforms the computer vision output into a collision-free reference trajectory. It was designed with modularity in mind, so that it is importable from anywhere. The module is spread accross multiple files for better organization and separation of concerns. The specific directory structure is shown below.

>```txt
>classes/gp/
>├── __init__.py     # init
>├── planner.py      # planning
>├── plotter.py      # plotting
>└── utils.py        # utilities
>```

The main API used for the project are the `GlobalPlanner` and `Plotter` classes. The former houses the general planning functionality, while the latter provides useful plots: either of generated paths, or comparison plots. Import of the aforementioned classes is, as already mentioned, accessible from anywhere within the project using the standard import syntax of the Python language:

In [1]:
# Module import (API used for the project)
from classes.gp import GlobalPlanner, Plotter

which exposes all the necessary functionality of global planning (and related plotting). This functionality is provided byt the `.plan()` method, owned by `GlobalPlanner`, and the `.path()` method, owned by `Plotter`. For more details, see the [here](./classes/gp/planner.py) and [here](./classes/gp/plotter.py).

### [2B] Independent Showcase

To show the independent functionality of the module, we provide dummy computer vision data below. Generally, the computer vision output (and thereby global planning input) obeys the following structure:

>```python
>cv_data = [
>    [W, H],                                     # map size (width, height)
>    np.array([[obs0_v0], ..., [obs0_vn]]),      # vertices of obstacle 0
>    ...,                                        # vertices of other obs.
>    np.array([[obsM_v0], ..., [obsM_vk]]),      # vertices of obstacle M
>    np.array([s_x, s_y]),                       # start coordinates (x,y)
>    np.array([g_x, g_y])                        # goal coordinates (x,y)
>]
>```

which can be changed to obtain different result, yet we need to be aware of edge cases where for some maps, no path can be found. This issue concerns only unreasonably designed maps, *i.e.*, those with very tight gaps, bad obstacle spacing, etc. All coordinates are in the computer vision frame. Using this information, the planner computes a reference path based on a visibility graph and an A* search.

In [2]:
import numpy as np

# Load "CV" data
data = [
        [260.0, 200.0],
        np.array([[60, 40], [90, 40], [90, 80], [60, 80]]),
        np.array([[130, 40], [200, 40], [200, 70], [130, 70]]),
        np.array([[120, 120], [160, 120], [160, 170], [120, 170]]),
        np.array([[40, 130], [70, 140], [65, 160], [35, 170], [25, 150]]),
        np.array([[200, 110], [230, 120], [220, 145]]),
        np.array([[180, 10], [230, 15], [225, 35], [175, 30]]),
        np.array([20, 20]),
        np.array([240, 180]),
    ]

Next, let us explore the aforementioned API. First, we need to initialize an instance of both classes. Besides the required `data` argument, the planner can also take additional arguments, namely the Thymio's radius and the spacing of intermediate goals along the reference path. We provide sample values below but note these *do not* represent the final values chosen. The final values can be found [here](./constants.py).

In [3]:
RADIUS = 2.0    # "Thymio radius" (circumscribed circle)
SPACING = 5.0   # Intermediate goals spacing

# Create GlobalPlanner instance
planner = GlobalPlanner(
    data,
    radius=RADIUS,
    spacing=SPACING
)

# Create Plotter instance
plotter = Plotter(planner)

Notice that the instance creation of `plotter` requires an instance of `planner`, as can again be seen [here](./classes/gp/plotter.py). With instances created, a reference path can be planned and plotted using the methods mentioned above. It is best to showcase the planning module through a series of plots which best highlight the stages it goes through. Note that these plots were generated on the sample data above and not the actual data. In the figure below, the initial stage produced is the original, plain course without any annotations from `planner`. Next, visibility is introduced (see middle figure), after which we finally generate our reference path using the A* algorithm.

The `GlobalPlanner` class (and its plotting counterpart) naturally extend to real Thymio runs. Besides the already mentioned API, we make use of the architecture in real runs in two other ways. Specifically, the `GlobalPlanner` and `Plotter` allow for:

* Moving the start and replanning from only this information. This functionality is achieved through the `start: Point = None` argument found in the `.plan()` method of `GlobalPlanner`. Specifically, we use this form of planning during kidnapping. We assume the environment Thymio navigates, *i.e.*, the obstacles and goal, stays constant in this state, except for the start.
* Overlaying the EKF estimate, if available, on top of the planned path

### [2C] Deep-Dive & Explanation